# Classification NBA Model

## Configuration

## Imports

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training
)
from sklearn.model_selection import KFold, cross_validate, train_test_split
from xgboost import XGBClassifier


## Load Data

In [94]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260227.csv"

path = data_path + name

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)
df_stats['GAME_DATE'] = pd.to_datetime(df_stats['GAME_DATE']).dt.strftime('%Y-%m-%d')

/tmp/ipykernel_723387/1910528255.py:6: DtypeWarning: Columns (666,667,716,717,765,766,814,815,863,864,912,913,1575,1576,1577,1625,1626,1627,1674,1675,1676,1723,1724,1725,1772,1773,1774,1821,1822,1823) have mixed types. Specify dtype option on import or set low_memory=False.
  df_stats = pd.read_csv(path)
/tmp/ipykernel_723387/1910528255.py:10: DtypeWarning: Columns (666,667,716,717,765,766,814,815,863,864,912,913,1575,1576,1577,1625,1626,1627,1674,1675,1676,1723,1724,1725,1772,1773,1774,1821,1822,1823) have mixed types. Specify dtype option on import or set low_memory=False.
  df_stats = pd.read_csv(


In [95]:
# #print column names with name in lower case
# for col in df_stats.columns:
#     if not col.isupper():
#         print(col)

In [96]:
#filter for last two seasons, gett seasons 2025 and 2024
# df_stats = df_stats[df_stats['SEASON_YEAR'].isin([2024, 2025])]

In [97]:
df_stats

,TOTAL_OVER_UNDER_LINE,TEAM_ID_TEAM_HOME,TEAM_CITY_TEAM_HOME,TEAM_ABBREVIATION_TEAM_HOME,TEAM_NAME_TEAM_HOME,MATCHUP_TEAM_HOME,GAME_NUMBER_TEAM_HOME,TEAM_ID_TEAM_AWAY,TEAM_CITY_TEAM_AWAY,TEAM_ABBREVIATION_TEAM_AWAY,...,TRAVEL_RECENCY_RATIO_AWAY_2D_OVER_14D,REST_DAYS_DIFF_HOME_MINUS_AWAY,INJURY_PTS_SHARE_HOME,INJURY_PTS_SHARE_AWAY,STAR_PTS_PCT_DIFF_HOME_MINUS_AWAY,POSS_X_TSPCT_HOME,POSS_X_TSPCT_AWAY,IS_WEEKEND,MONTH,IS_US_HOLIDAY
0,237.5,1610612742,Dallas,DAL,Dallas Mavericks,DAL vs. MEM,59,1610612763,Memphis,MEM,...,0.000000,-1,0.651722,0.875932,0.045164,58.047968,57.341333,0,2,0
1,231.5,1610612760,Oklahoma City,OKC,Oklahoma City Thunder,OKC vs. DEN,61,1610612743,Denver,DEN,...,0.859433,0,0.299739,0.355000,0.035547,59.363195,61.368010,0,2,0
2,223.5,1610612749,Milwaukee,MIL,Milwaukee Bucks,MIL vs. NYK,58,1610612752,New York,NYK,...,0.000000,-1,0.325247,0.172359,-0.053131,59.580380,57.233608,0,2,0
3,208.5,1610612738,Boston,BOS,Boston Celtics,BOS vs. BKN,59,1610612751,Brooklyn,BKN,...,0.000000,1,0.009315,0.100015,0.025048,55.043007,49.916071,0,2,0
4,225.5,1610612765,Detroit,DET,Detroit Pistons,DET vs. CLE,58,1610612739,Cleveland,CLE,...,0.752393,0,0.062965,0.513804,0.070626,54.091756,59.851827,0,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11107,NaN,1610612758,Sacramento,SAC,Sacramento Kings,SAC vs. HOU,1,1610612745,Houston,HOU,...,NaN,-1,NaN,0.000000,NaN,NaN,58.479000,0,10,0
11108,NaN,1610612764,Washington,WAS,Washington Wizards,WAS vs. PHI,1,1610612755,Philadelphia,PHI,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0,10,0
11109,NaN,1610612754,Indiana,IND,Indiana Pacers,IND vs. BKN,1,1610612751,Brooklyn,BKN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0,10,0
11110,NaN,1610612739,Cleveland,CLE,Cleveland Cavaliers,CLE vs. BOS,1,1610612738,Boston,BOS,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0,10,0


In [98]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=5, drop_all_na_rows=True, create_missing_flags=False, verbose=1)

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 11112 rows
Basic cleaning complete: 8505 rows remaining

Starting advanced column cleaning with 2139 columns

Advanced column cleaning complete: 2139 → 1122 columns (1017 removed)


Dropping NA rows for SEASON_YEAR 2017...
   Removed 0 rows with NaN values from 2017 season

Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 11 (0.13%)
  Critical columns requiring data: 5
  Columns zero-filled: 210
  Infer pairs applied: 8/264
  Remaining NaN cells: 51505

Dropping rows that contain NaN...
CLEANING COMPLETE
Final shape: (7312, 1122)


In [99]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_stats[df_stats[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_stats.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year


In [100]:
BET365_LINE_COL =  "total_bet365_line_over"

# Ensure scoring line and target exist (avoid NaN-driven undefined betting accuracy).
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()

In [101]:
df_to_train['LINE_ERROR'] = df_to_train['TOTAL_POINTS'] - df_to_train['TOTAL_OVER_UNDER_LINE']
# df_to_train['LINE_ERROR'] = df_to_train['TOTAL_POINTS'] - df_to_train[BET365_LINE_COL]

## Train / Test

In [102]:
X = df_to_train.drop(['TOTAL_POINTS', 'LINE_ERROR', 'SEASON_YEAR'], axis=1, errors='ignore')
y = df_to_train['LINE_ERROR']

In [103]:
# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=16)

In [104]:
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
# Check number of coulmns
print(f"Number of columns in training set: {X_train.shape[1]}")
print(f"Number of columns in test set: {X_test.shape[1]}")

Training set size: 6215
Test set size: 1097
Number of columns in training set: 1120
Number of columns in test set: 1120


## Cross-validation

In [105]:
from sklearn.model_selection import KFold, cross_validate, cross_val_predict, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, make_scorer, root_mean_squared_error


In [106]:
# Declare KFold
kf = KFold(n_splits=5, shuffle=True, random_state=16)

In [107]:
import numpy as np


def over_under_betting_accuracy(true_error: np.ndarray, pred_error: np.ndarray) -> float:
    true_sign = np.sign(true_error)
    pred_sign = np.sign(pred_error)

    valid = (true_sign != 0) & (pred_sign != 0)
    if not np.any(valid):
        return np.nan

    return float(np.mean(true_sign[valid] == pred_sign[valid]))

class OverUnderScorer:
    """
    Custom sklearn-compatible scorer for over/under betting accuracy.
    """

    def __call__(self, estimator, X, y_true):
        y_pred = estimator.predict(X)

        # betting_line = X["TOTAL_OVER_UNDER_LINE"].to_numpy(dtype=float)

        return over_under_betting_accuracy(
            true_error=np.asarray(y_true, dtype=float),
            pred_error=np.asarray(y_pred, dtype=float),
            # betting_line=betting_line,
        )
over_under_scorer = OverUnderScorer()


In [108]:
# Declare scores to be used
scoring = {
    'MSE': make_scorer(mean_squared_error),
    'RMSE': make_scorer(root_mean_squared_error),
    'MAE': make_scorer(mean_absolute_error),
    'OU_Betting_Accuracy': over_under_scorer,
}

In [109]:
def print_metrics(cv_results):
    for sc in scoring.keys():
        if sc == 'OU_Betting_Accuracy':
            print(f'Train {sc}:', f"{cv_results[f'train_{sc}'].mean():.2%}")
            print(f'Validation {sc}:', f"{cv_results[f'test_{sc}'].mean():.2%}")
        else:
            print(f'Train {sc}:', cv_results[f'train_{sc}'].mean().round(5))
            print(f'Validation {sc}:', cv_results[f'test_{sc}'].mean().round(5))
        print()

In [110]:
def real_vs_pred(model, X_train, y_train):
    preds = cross_val_predict(model, X_train, y_train, cv=kf, n_jobs=-1)
    x_line = np.arange(y_train.min(), y_train.max())
    plt.scatter(y_train, preds)
    plt.plot(x_line, x_line, color='orange')
    plt.xlabel('Real target')
    plt.ylabel('Predicted target')
    plt.show()

## Baseline

In [111]:
from sklearn.dummy import DummyRegressor

season_bl = DummyRegressor(strategy='mean')
cv_results = cross_validate(season_bl, X_train, y_train, cv=kf,
                            scoring=scoring, return_train_score=True)
season_bl.fit(X_train, y_train)
print_metrics(cv_results)

Train MSE: 309.48793
Validation MSE: 309.61515

Train RMSE: 17.59212
Validation RMSE: 17.59348

Train MAE: 14.02588
Validation MAE: 14.02768

Train OU_Betting_Accuracy: 52.88%
Validation OU_Betting_Accuracy: 52.88%



In [112]:
# Baseline 3: Predict the betting line (TOTAL_OVER_UNDER_LINE)
y_pred_baseline_3 = X_train['TOTAL_OVER_UNDER_LINE']

# Evaluate
mse = mean_squared_error(y_train, y_pred_baseline_3)
mae = mean_absolute_error(y_train, y_pred_baseline_3)
rmse = root_mean_squared_error(y_train, y_pred_baseline_3)
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")

MSE: 51949.34
RMSE: 227.92
MAE: 227.06


## Logistic Regression

In [113]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
cv_results = cross_validate(lr, X_train.fillna(0), y_train, cv=kf,
                            scoring=scoring, return_train_score=True)

print_metrics(cv_results)

Train MSE: 223.31368
Validation MSE: 421.16727

Train RMSE: 14.94349
Validation RMSE: 20.46716

Train MAE: 11.85629
Validation MAE: 15.62866

Train OU_Betting_Accuracy: 67.02%
Validation OU_Betting_Accuracy: 53.56%



In [123]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor

# Example XGBoost regressor:
xgb_reg = XGBRegressor(
    max_depth=4,
    learning_rate=0.02,
    n_estimators=500,
    subsample=0.5,       # Equivalent to max_samples in GBRegressor
    colsample_bytree=0.6, # Equivalent to max_features in GBRegressor
    n_jobs=-2,
    random_state=16)

cv_results = cross_validate(
    xgb_reg, 
    X_train, y_train, 
    cv=kf, 
    scoring=scoring,      # Use your custom scoring or e.g. 'neg_mean_absolute_error'
    return_train_score=True,
    n_jobs=-2
)
# Train final model on full train set
xgb_reg.fit(X_train, y_train)

# Print metrics
print_metrics(cv_results)


Train MSE: 167.07607
Validation MSE: 299.56863

Train RMSE: 12.92563
Validation RMSE: 17.30586

Train MAE: 10.31597
Validation MAE: 13.78701

Train OU_Betting_Accuracy: 82.03%
Validation OU_Betting_Accuracy: 54.99%



In [124]:
feature_importances = xgb_reg.feature_importances_
important_features = np.argsort(feature_importances)[::-1]  
feature_importances = pd.DataFrame({
    'Feature': X_train.columns[important_features],
    'Importance': feature_importances[important_features]
}).sort_values(by="Importance", ascending=False)


selected_features = feature_importances[feature_importances['Importance'] > 0]['Feature'].tolist()
feature_importances


,Feature,Importance
0,TOTAL_LINE_draftkings_LAST_HOME_AWAY_10_WMA_BE...,0.004402
1,TOTAL_LINE_draftkings_SEASON_BEFORE_AVG_TEAM_HOME,0.002130
2,TOTAL_LINE_caesars_SEASON_BEFORE_AVG_TEAM_HOME,0.002106
3,odds_ml_away_prob_novig_fanduel,0.002086
4,TOTAL_LINE_fanduel_LAST_HOME_AWAY_5_WMA_BEFORE...,0.002067
...,...,...
1067,DIFF_FROM_LINE_bet365_LAST_HOME_AWAY_5_MATCHES...,0.000000
1068,TOP6_INJURED_STREAK_PTS_BEFORE_TEAM_HOME,0.000000
1069,TOP5_INJURED_STREAK_PTS_BEFORE_TEAM_HOME,0.000000
1102,TOTAL_LINE_caesars_LAST_ALL_5_MATCHES_BEFORE_T...,0.000000


In [125]:
import numpy as np
import pandas as pd


def error_sign_accuracy(y_true_error, y_pred_error) -> float:
    """
    Same sign of (predicted error) and (true error).

    Push handling:
    - If either sign is 0, exclude that sample from scoring.
      (If you prefer to count (0,0) as correct, I can change it.)
    """
    y_true_error = np.asarray(y_true_error, dtype=float)
    y_pred_error = np.asarray(y_pred_error, dtype=float)

    true_sign = np.sign(y_true_error)
    pred_sign = np.sign(y_pred_error)

    valid = (true_sign != 0) & (pred_sign != 0)
    if not np.any(valid):
        return np.nan

    return float(np.mean(true_sign[valid] == pred_sign[valid]))


def evaluate_error_thresholds(
    model,
    X_test: pd.DataFrame,
    y_test_error: pd.Series,
    thresholds=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10),
):
    """
    Evaluate directional accuracy at different confidence thresholds.

    Threshold rule:
    - Include a game if abs(predicted_error) > t
    """
    # Predict error directly
    y_pred_error = np.asarray(model.predict(X_test), dtype=float)

    margin = np.abs(y_pred_error)

    rows = []
    n_total = len(y_test_error)

    y_true_error_np = y_test_error.to_numpy(dtype=float)

    for t in thresholds:
        mask = margin > t
        n = int(mask.sum())

        acc = (
            np.nan
            if n == 0
            else error_sign_accuracy(y_true_error_np[mask], y_pred_error[mask])
        )

        rows.append(
            {
                "threshold_abs_pred_error_gt": t,
                "n_games": n,
                "pct_of_test": (n / n_total) if n_total else np.nan,
                "directional_accuracy": acc,
            }
        )

    return pd.DataFrame(rows), y_pred_error
results_df, y_pred_test_error = evaluate_error_thresholds(
    model=xgb_reg,
    X_test=X_test,
    y_test_error=y_test,      # y_test must be the REAL ERROR series
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "directional_accuracy": "{:.2%}"}
    )
)



,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,0,1097,100.0%,53.19%
1,1,845,77.0%,55.15%
2,2,626,57.1%,57.99%
3,3,434,39.6%,59.68%
4,4,285,26.0%,60.70%
5,5,163,14.9%,63.19%
6,6,103,9.4%,66.02%
7,7,64,5.8%,70.31%
8,8,39,3.6%,74.36%
9,9,27,2.5%,81.48%


### Selected Features

In [128]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor

# Example XGBoost regressor:
xgb_reg_important = XGBRegressor(
    max_depth=4,
    learning_rate=0.02,
    n_estimators=350,
    subsample=0.5,       # Equivalent to max_samples in GBRegressor
    colsample_bytree=0.6, # Equivalent to max_features in GBRegressor
    n_jobs=-2,
    random_state=16)

cv_results_important = cross_validate(
    xgb_reg, 
    X_train[selected_features], y_train, 
    cv=kf, 
    scoring=scoring,      # Use your custom scoring or e.g. 'neg_mean_absolute_error'
    return_train_score=True,
    n_jobs=-2
)
# Train final model on full train set
xgb_reg_important.fit(X_train[selected_features], y_train)

# Print metrics
print_metrics(cv_results_important)


Train MSE: 166.79092
Validation MSE: 299.76086

Train RMSE: 12.91466
Validation RMSE: 17.31117

Train MAE: 10.30373
Validation MAE: 13.79728

Train OU_Betting_Accuracy: 82.39%
Validation OU_Betting_Accuracy: 55.88%



In [130]:
results_df, y_pred_test_error = evaluate_error_thresholds(
    model=xgb_reg_important,
    X_test=X_test[selected_features],
    y_test_error=y_test,      # y_test must be the REAL ERROR series
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "directional_accuracy": "{:.2%}"}
    )
)

,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,0,1097,100.0%,55.02%
1,1,799,72.8%,56.52%
2,2,560,51.0%,57.86%
3,3,371,33.8%,57.95%
4,4,210,19.1%,61.90%
5,5,116,10.6%,64.66%
6,6,67,6.1%,70.15%
7,7,44,4.0%,77.27%
8,8,23,2.1%,91.30%
9,9,16,1.5%,93.75%


# Optuna

In [ ]:
import numpy as np
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from xgboost import XGBRegressor
from sklearn.model_selection import KFold

# LINE_COL = "TOTAL_OVER_UNDER_LINE"


def over_under_betting_accuracy(true_error: np.ndarray, pred_error: np.ndarray) -> float:
    true_sign = np.sign(true_error)
    pred_sign = np.sign(pred_error)

    valid = (true_sign != 0) & (pred_sign != 0)
    if not np.any(valid):
        return np.nan

    return float(np.mean(true_sign[valid] == pred_sign[valid]))


def ou_accuracy_with_threshold_errors(
    true_error: np.ndarray,
    pred_error: np.ndarray,
    threshold_abs: float = 0.0,
    min_coverage: float = 0.25,
):
    # "Edge" in points relative to the line
    margin = np.abs(pred_error)

    # Bet only when predicted edge exceeds threshold
    mask = margin > threshold_abs
    coverage = float(np.mean(mask))

    if coverage < min_coverage or mask.sum() == 0:
        return 0.0, coverage

    score = over_under_betting_accuracy(true_error[mask], pred_error[mask])

    # If everything got filtered as pushes (rare, but possible), penalize
    if np.isnan(score):
        return 0.0, coverage

    return score, coverage

def _predict_best(model, X):
    # Use best iteration if early stopping happened
    if getattr(model, "best_iteration", None) is not None:
        # Newer XGBoost
        try:
            return model.predict(X, iteration_range=(0, model.best_iteration + 1))
        except TypeError:
            # Older compatibility path
            ntree_limit = getattr(model, "best_ntree_limit", None)
            if ntree_limit is not None:
                return model.predict(X, ntree_limit=ntree_limit)
    return model.predict(X)

def objective(trial, X, y):
    threshold_abs = trial.suggest_float("threshold_abs", 0.0, 10.0, step=0.5)
    min_coverage = 0.25

    params = {
        "booster": "gbtree",
        "tree_method": "hist",
        "max_depth": trial.suggest_int("max_depth", 2, 7),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-2, 30.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "subsample": trial.suggest_float("subsample", 0.3, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "learning_rate": trial.suggest_float("learning_rate", 0.0075, 0.2, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-6, 50.0, log=True),

        "n_estimators": trial.suggest_int("n_estimators", 225, 450, step=25),
        "early_stopping_rounds": 200,   # moved here
        "eval_metric": "rmse",          # optional, but explicit is good
        "random_state": 16,
        "n_jobs": -1,
    }

    fold_scores = []
    fold_coverages = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr = y.iloc[tr_idx].to_numpy()
        y_va = y.iloc[va_idx].to_numpy()

        model = XGBRegressor(**params)

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False,
        )

        y_pred_error = _predict_best(model, X_va)

        score, coverage = ou_accuracy_with_threshold_errors(
            true_error=y_va,
            pred_error=y_pred_error,
            threshold_abs=threshold_abs,
            min_coverage=min_coverage,
        )

        fold_scores.append(score)
        fold_coverages.append(coverage)

        trial.report(float(np.mean(fold_scores)), step=fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    trial.set_user_attr("mean_coverage", float(np.mean(fold_coverages)))
    return float(np.mean(fold_scores))

# ----------------------------
# Run the search (2 to 3 hours)
# ----------------------------
# Make sure X_train includes TOTAL_OVER_UNDER_LINE, since you use it in the metric.
# X_train, y_train are your current train split.
study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=16),
    pruner=MedianPruner(n_warmup_steps=2),
)

# Time budget: 3 hours. You can adjust to 2*3600 if you want.
study.optimize(lambda t: objective(t, X_train, y_train), timeout= 2*3600, n_jobs=1)

print("Best value (CV OU success):", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(k, v)

# ----------------------------
# Train final model on full train set
# ----------------------------
best_params = study.best_params.copy()
best_threshold = best_params.pop("threshold_abs")  # strategy threshold

final_params = {
    "booster": "gbtree",
    "tree_method": "hist",
    "random_state": 16,
    "n_jobs": -1,
    **best_params,
}

final_model = XGBRegressor(**final_params)

[I 2026-02-20 00:27:32,964] A new study created in memory with name: no-name-85480779-8e4a-4b2b-b8cc-8443cb3bb675
[I 2026-02-20 00:28:50,308] Trial 0 finished with value: 0.3622081913067786 and parameters: {'threshold_abs': 2.0, 'max_depth': 5, 'min_child_weight': 0.8219695696031055, 'gamma': 0.22800975066403162, 'subsample': 0.5525101847434408, 'colsample_bytree': 0.5338485650147733, 'learning_rate': 0.071971944311379, 'reg_alpha': 2.975656697106555e-07, 'reg_lambda': 3.478796625225186e-06, 'n_estimators': 450}. Best is trial 0 with value: 0.3622081913067786.
[I 2026-02-20 00:29:29,516] Trial 1 finished with value: 0.0 and parameters: {'threshold_abs': 5.5, 'max_depth': 2, 'min_child_weight': 3.2561820715318097, 'gamma': 0.7922608676158321, 'subsample': 0.4751969147208164, 'colsample_bytree': 0.5760923536613185, 'learning_rate': 0.07385949950651723, 'reg_alpha': 0.0001507914760359121, 'reg_lambda': 4.526442350789216e-05, 'n_estimators': 325}. Best is trial 0 with value: 0.362208191306

Best value (CV OU success): 0.5902098508971209
Best params:
threshold_abs 2.0
max_depth 5
min_child_weight 0.41186206628258415
gamma 1.3295724600453376
subsample 0.4081456932920822
colsample_bytree 0.6737445297595096
learning_rate 0.02190111316215935
reg_alpha 9.638387046827754e-06
reg_lambda 0.011165431717790846
n_estimators 450


## Check in Test set

In [28]:
final_model.fit(X_train, y_train, verbose=False)
results_df_final, y_pred_test = evaluate_error_thresholds(
    model=final_model,
    X_test=X_test,
    y_test_error=y_test,
    thresholds=range(0, 11)  # 0..10
)

# Pretty display
display(results_df_final.style.format({
    "pct_of_test": "{:.1%}",
    "error_betting_accuracy": "{:.2%}",
}))


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,0,1080,100.0%,0.545370
1,1,855,79.2%,0.563164
2,2,641,59.4%,0.563780
3,3,453,41.9%,0.564444
4,4,323,29.9%,0.588785
5,5,214,19.8%,0.605634
6,6,134,12.4%,0.626866
7,7,89,8.2%,0.696629
8,8,49,4.5%,0.714286
9,9,36,3.3%,0.694444


In [29]:
feature_importances = final_model.feature_importances_
important_features = np.argsort(feature_importances)[::-1]  
feature_importances = pd.DataFrame({
    'Feature': X_train.columns[important_features],
    'Importance': feature_importances[important_features]
}).sort_values(by="Importance", ascending=False)
feature_importances

,Feature,Importance
0,total_draftkings_price_over,0.003485
1,odds_book_total_line_move_from_opener_fanduel,0.003007
2,MONEYLINE_TEAM_AWAY,0.002346
3,odds_book_total_line_move_from_opener_draftkings,0.002277
4,TOTAL_POINTS_SEASON_BEFORE_AVG_TEAM_HOME,0.002270
...,...,...
891,ml_caesars_price_home,0.000000
892,BACK_TO_BACK,0.000000
893,TOP4_INJURED_PLAYER_PTS_BEFORE_TEAM_AWAY,0.000000
906,odds_spread_home_is_fav_draftkings,0.000000


In [30]:

import json
from pathlib import Path

import joblib
import numpy as np
from xgboost import XGBRegressor


def train_final_model(
    X_all,
    y_all,
    study,
    *,
    use_early_stopping: bool = True,
    val_frac: float = 0.1,
    random_state: int = 16,
):
    """
    Train a final XGBRegressor using Optuna best params.
    Also returns the betting threshold (threshold_abs) stored in the study params.
    """
    best_params = study.best_params.copy()
    best_threshold = float(best_params.pop("threshold_abs"))

    final_params = {
        "booster": "gbtree",
        "tree_method": "hist",
        "random_state": random_state,
        "n_jobs": -1,
        "eval_metric": "rmse",
        **best_params,
    }

    model = XGBRegressor(**final_params)

    if not use_early_stopping:
        # Train on all data without early stopping
        model.fit(X_all, y_all, verbose=False)
        return model, best_threshold

    # Keep early stopping by holding out a small validation split
    n = len(X_all)
    rng = np.random.default_rng(random_state)
    idx = np.arange(n)
    rng.shuffle(idx)

    n_val = max(1, int(round(val_frac * n)))
    val_idx = idx[:n_val]
    tr_idx = idx[n_val:]

    X_tr, y_tr = X_all.iloc[tr_idx], y_all.iloc[tr_idx]
    X_va, y_va = X_all.iloc[val_idx], y_all.iloc[val_idx]

    model.set_params(early_stopping_rounds=200)
    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False,
    )
    return model, best_threshold


def save_model_bundle(
    model: XGBRegressor,
    threshold_abs: float,
    feature_names: list[str],
    out_dir: str | Path,
    name: str = "xgb_ou_model",
):
    """
    Saves:
      - model.joblib (sklearn wrapper)
      - metadata.json (threshold + feature order)
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    model_path = out_dir / f"{name}.joblib"
    meta_path = out_dir / f"{name}.meta.json"

    joblib.dump(model, model_path)

    metadata = {
        "threshold_abs": float(threshold_abs),
        "feature_names": list(feature_names),
        "xgboost_version_note": "Saved via joblib XGBRegressor wrapper",
    }
    meta_path.write_text(json.dumps(metadata, indent=2))

    return model_path, meta_path


def load_model_bundle(model_path: str | Path, meta_path: str | Path):
    model = joblib.load(model_path)
    metadata = json.loads(Path(meta_path).read_text())
    return model, metadata


# -------------------------
# Example usage
# -------------------------

# 1) Build ALL data (replace with your variables)
X_all = X  # your full feature DF, must include TOTAL_OVER_UNDER_LINE column for later scoring/decision rule
y_all = y  # your full target Series

# 2) Train final model
final_model, best_threshold = train_final_model(
    X_all=X_all,
    y_all=y_all,
    study=study,
    use_early_stopping=True,   # or False if you prefer no holdout
    val_frac=0.1,
    random_state=16,
)

# 3) Save model + metadata (threshold + feature ordering)
model_path, meta_path = save_model_bundle(
    model=final_model,
    threshold_abs=best_threshold,
    feature_names=list(X_all.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/dif_to_line/",
    name="drop_na_cols_5_all_data_xgb_line_error_14_02_26",
)

print("Saved:", model_path)
print("Saved:", meta_path)

# 4) Load later
loaded_model, meta = load_model_bundle(model_path, meta_path)
print("Loaded threshold_abs:", meta["threshold_abs"])


Saved: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/dif_to_line/drop_na_cols_5_all_data_xgb_line_error_14_02_26.joblib
Saved: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/dif_to_line/drop_na_cols_5_all_data_xgb_line_error_14_02_26.meta.json
Loaded threshold_abs: 2.0
